# 01 — Baseline YOLOv8n on VisDrone

This notebook trains a baseline YOLOv8n model on a subset of the VisDrone2019-DET dataset. The goal is to establish a reference mAP score before adding CBAM attention or pseudo-labeling.

**Expected mAP50:** 0.20 - 0.30 (small aerial objects are inherently difficult to detect).

## Install Dependencies

In [1]:
!pip install ultralytics kaggle -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:00


## Mount Google Drive and Set Working Directory

In [2]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

candidate_paths = [
    Path('/content/drive/MyDrive/uav-small-object-detector'),
    Path('/content/drive/MyDrive/uav-small-object-detector/UAV_small_obj_detector'),
    Path('/content/drive/MyDrive/UAV_small_obj_detector'),
]

project_root = next((path for path in candidate_paths if path.exists()), None)
if project_root is None:
    raise FileNotFoundError(
        'Could not find the project folder in Google Drive. '
        'Check MyDrive/uav-small-object-detector and MyDrive/UAV_small_obj_detector.'
    )

os.chdir(project_root)
print(f'Working directory: {project_root}')
!pwd

Mounted at /content/drive
Working directory: /content/drive/MyDrive/UAV_small_obj_detector
/content/drive/MyDrive/UAV_small_obj_detector


## Download VisDrone2019-DET Dataset

In [ ]:
# Download VisDrone directly using Ultralytics' maintained dataset assets.
from pathlib import Path
import os
import shutil
from PIL import Image
from ultralytics.utils.downloads import download

dataset_root = Path('/content/VisDrone')

def visdrone2yolo(root, split, source_name):
    source_dir = root / source_name
    images_dir = root / 'images' / split
    labels_dir = root / 'labels' / split
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for img in (source_dir / 'images').glob('*.jpg'):
        shutil.move(str(img), str(images_dir / img.name))

    for ann in (source_dir / 'annotations').glob('*.txt'):
        img_path = images_dir / ann.with_suffix('.jpg').name
        img_w, img_h = Image.open(img_path).size
        dw, dh = 1.0 / img_w, 1.0 / img_h
        lines = []

        with open(ann, encoding='utf-8') as f:
            for row in [x.split(',') for x in f.read().strip().splitlines()]:
                if row[4] != '0':
                    x, y, w, h = map(int, row[:4])
                    cls = int(row[5]) - 1
                    x_center = (x + w / 2) * dw
                    y_center = (y + h / 2) * dh
                    w_norm = w * dw
                    h_norm = h * dh
                    lines.append(f'{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n')

        with open(labels_dir / ann.name, 'w', encoding='utf-8') as out:
            out.writelines(lines)

urls = [
    'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
    'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
    'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip',
]
download(urls, dir=dataset_root, threads=4)

for folder, split in {
    'VisDrone2019-DET-train': 'train',
    'VisDrone2019-DET-val': 'val',
    'VisDrone2019-DET-test-dev': 'test',
}.items():
    visdrone2yolo(dataset_root, split, folder)
    shutil.rmtree(dataset_root / folder)

print(f"Train images found: {len(os.listdir('/content/VisDrone/images/train'))}")
print(f"Validation images found: {len(os.listdir('/content/VisDrone/images/val'))}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Unzipping /content/VisDrone/VisDrone2019-DET-val.zip to /content/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 429.9files/s 2.6s
Unzipping /content/VisDrone/VisDrone2019-DET-test-dev.zip to /content/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 315.6files/s 10.2s
Unzipping /content/VisDrone/VisDrone2019-DET-train.zip to /content/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 1.6Kfiles/s 8.0s
Train images found: 6471
Validation images found: 548


## Create VisDrone YAML Configuration

In [4]:
%%writefile visdrone.yaml
path: /content/VisDrone
train: images/train
val: images/val
nc: 10
names: ['pedestrian','people','bicycle','car','van','truck',
        'tricycle','awning-tricycle','bus','motor']

Overwriting visdrone.yaml


## Create Output Directories

In [5]:
import os
os.makedirs('results/detection_samples', exist_ok=True)
os.makedirs('results/gradcam_samples', exist_ok=True)
os.makedirs('src', exist_ok=True)

## Load Pretrained YOLOv8n and Train

In [6]:
from ultralytics import YOLO

# Load pretrained weights
model = YOLO('yolov8n.pt')

# Fine-tune on VisDrone
results = model.train(
    data='visdrone.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    name='baseline_visdrone'
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=visdrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_visdrone2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.

## Evaluate on Validation Split

In [7]:
metrics = model.val()
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2526.0±475.6 MB/s, size: 126.7 KB)
val: Scanning /content/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 209.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.8it/s 12.3s
                   all        548      38759      0.392      0.303      0.294      0.166
            pedestrian        520       8844      0.391      0.326      0.309      0.127
                people        482       5125      0.444      0.222      0.241     0.0823
               bicycle        364       1287      0.178     0.0903      0.056     0.0187
                   car        515      14064      0.546      0.744      0.721       0.48
                   van        421       1975   

## Save Metrics to JSON

In [8]:
import json

metrics_data = {
    "baseline": {
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr)
    }
}

with open('results/metrics.json', 'w') as f:
    json.dump(metrics_data, f, indent=2)

print("Metrics saved to results/metrics.json")
print(json.dumps(metrics_data, indent=2))

Metrics saved to results/metrics.json
{
  "baseline": {
    "mAP50": 0.29373669317784457,
    "mAP50-95": 0.1662671086264691,
    "precision": 0.39159907325051424,
    "recall": 0.30258304542539205
  }
}


## Save 5 Sample Detection Images

In [10]:
import os
import shutil

val_images = sorted(os.listdir('/content/VisDrone/images/val'))
for i in range(5):
    src = f"/content/VisDrone/images/val/{val_images[i]}"
    dst = f"results/detection_samples/sample_{i}.jpg"
    shutil.copy(src, dst)

print("Sample images saved to results/detection_samples/")

Sample images saved to results/detection_samples/


## Results Summary

**Baseline YOLOv8n (20 epochs on VisDrone)**

| Metric | Value |
|---|---|
| mAP@50 | 0.294 |
| mAP@50-95 | 0.166 |
| Precision | 0.392 |
| Recall | 0.303 |

**Next step:** Open `02_cbam_attention.ipynb` to add CBAM attention and compare mAP scores.